# RLHFlow PRM — System Prompt Effect on Scores

Examine how adding a system message affects
`RLHFlow/Llama3.1-8B-PRM-Deepseek-Data` per-step scores.
Uses the inline `score_rlhflow_prm` function (no wrapper)
so a single model load can be reused across all prompts.

The baseline RLHFlow conversation has no system turn; the
scoring function accepts an optional `system` argument that
prepends a system message to both the scoring and marker
conversations when non-empty.

Two examples scored under each system prompt:
1. Flamingo problem (all steps correct)
2. Algebra correct vs. wrong trajectory

Env: py311 / transformers 4.57, fp16 on V100.

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)
logging.disable(logging.CRITICAL)

import gc
import sys
sys.path.append("..")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from notebook_utils import gpu_mem_used_gb, print_step_scores
from utils.configs import system_prompt as genconfig_system_prompt

base_dir = "/groups/chichengz/tnn/datasets"
rlhflow_prm_dir = f"{base_dir}/Llama3.1-8B-PRM-Deepseek-Data"

## System prompts

Four variants to compare:
- **empty** — no system message (baseline, matches original
  RLHFlow usage)
- **qwen_sys_prompt** — Qwen model-card prompt (for
  cross-model comparison)
- **short** — minimal instruction, no boxed-answer requirement
- **custom_sys_prompt** — `GenConfig.system_prompt` from
  `utils/configs.py`

In [2]:
system_prompts = {
    "empty": "",
    "qwen_sys_prompt": (
        "Please reason step by step, and put your final answer "
        "within \\boxed{}."
    ),
    "custom_sys_prompt": genconfig_system_prompt,
    "short": "Solve the problem step by step.",
}

## Scoring function

The conversation alternates user reasoning-step messages with
assistant `+` judgements. A parallel conversation swaps `+`
for the unique marker token `ки` to locate each score
position. When `system` is non-empty, a system turn is
prepended to both conversations. Set `print_conversation=True`
to dump the full formatted PRM input.

In [3]:
def score_rlhflow_prm(
    model,
    tokenizer,
    problem: str,
    steps: list[str],
    candidate_token_ids: list[int],
    system: str = "",
    print_conversation: bool = False,
) -> list[float]:
    # marker_text "ки" is a Cyrillic bigram that maps to a
    # single unique vocab token; the parallel marker
    # conversation lets us find where each assistant `+` sits.
    marker_text = "ки"
    marker_token_id = (
        tokenizer(marker_text, return_tensors="pt")
        .input_ids[0, 1].item()
    )

    sys_turn = [{"role": "system", "content": system}] \
        if system else []

    conversation = list(sys_turn)
    marker_conversation = list(sys_turn)
    step_scores = []

    for step_idx, step in enumerate(steps):
        text = (problem + " " + step) if step_idx == 0 else step

        conversation.append({"role": "user", "content": text})
        conversation.append({"role": "assistant", "content": "+"})

        marker_conversation.append(
            {"role": "user", "content": text}
        )
        marker_conversation.append(
            {"role": "assistant", "content": marker_text}
        )

        input_ids = tokenizer.apply_chat_template(
            conversation, return_tensors="pt",
        ).to(model.device)
        marker_input_ids = tokenizer.apply_chat_template(
            marker_conversation, return_tensors="pt",
        ).to(model.device)

        if input_ids.shape != marker_input_ids.shape:
            raise RuntimeError(
                f"Marker conversation shape mismatch: "
                f"{input_ids.shape} vs {marker_input_ids.shape}"
            )

        with torch.no_grad():
            logits = model(input_ids=input_ids).logits[
                :, :, candidate_token_ids
            ]
            probs = logits.softmax(dim=-1)[:, :, 0]

        # The model predicts token N from position N-1. Locate
        # the marker token and read the preceding position's
        # P(+). Use the last marker for the current step.
        marker_positions = (
            marker_input_ids[0, 1:] == marker_token_id
        ).nonzero(as_tuple=True)[0]
        if marker_positions.numel() != step_idx + 1:
            raise RuntimeError(
                f"Expected {step_idx + 1} marker positions, "
                f"found {marker_positions.numel()}"
            )
        score_pos = marker_positions[-1].item()
        step_scores.append(
            probs[0, score_pos].detach().cpu().float().item()
        )

    if print_conversation:
        full = tokenizer.apply_chat_template(
            conversation, tokenize=False,
        )
        print("===== Formatted PRM input =====")
        print(full)
        print("===============================")

    return step_scores

## Load the PRM

In [4]:
tokenizer = AutoTokenizer.from_pretrained(rlhflow_prm_dir)
model = AutoModelForCausalLM.from_pretrained(
    rlhflow_prm_dir,
    device_map="cuda:0",
    dtype=torch.float16,
).eval()

# Llama ships no pad token; reuse EOS for batched calls.
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

# encode("+") prepends BOS; [-1] picks the actual "+" id.
plus_token_id = tokenizer.encode("+")[-1]
minus_token_id = tokenizer.encode("-")[-1]
candidate_token_ids = [plus_token_id, minus_token_id]

print(f"candidate token ids: {candidate_token_ids}")
print(f"dtype              : {next(model.parameters()).dtype}")
print(f"GPU memory used    : {gpu_mem_used_gb():.2f} GB")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

candidate token ids: [10, 12]
dtype              : torch.float16
GPU memory used    : 16.41 GB


## Example 1 — flamingo problem

All four steps are correct. We expect high scores throughout;
the question is whether adding a system prompt shifts the
absolute values or changes the relative ranking.

In [5]:
problem = (
    "Sue lives in a fun neighborhood.  One weekend, the "
    "neighbors decided to play a prank on Sue.  On Friday "
    "morning, the neighbors placed 18 pink plastic flamingos "
    "out on Sue's front yard.  On Saturday morning, the "
    "neighbors took back one third of the flamingos, painted "
    "them white, and put these newly painted white flamingos "
    "back out on Sue's front yard.  Then, on Sunday morning, "
    "they added another 18 pink plastic flamingos to the "
    "collection. At noon on Sunday, how many more pink "
    "plastic flamingos were out than white plastic flamingos?"
)

reasoning_steps = [
    "To find out how many more pink plastic flamingos were "
    "out than white plastic flamingos at noon on Sunday, we "
    "can break down the problem into steps. First, on Friday, "
    "the neighbors start with 18 pink plastic flamingos.",

    "On Saturday, they take back one third of the flamingos. "
    "Since there were 18 flamingos, (1/3 \\times 18 = 6) "
    "flamingos are taken back. So, they have (18 - 6 = 12) "
    "flamingos left in their possession. Then, they paint "
    "these 6 flamingos white and put them back out on Sue's "
    "front yard. Now, Sue has the original 12 pink flamingos "
    "plus the 6 new white ones. Thus, by the end of Saturday, "
    "Sue has (12 + 6 = 18) pink flamingos and 6 white "
    "flamingos.",

    "On Sunday, the neighbors add another 18 pink plastic "
    "flamingos to Sue's front yard. By the end of Sunday "
    "morning, Sue has (18 + 18 = 36) pink flamingos and "
    "still 6 white flamingos.",

    "To find the difference, subtract the number of white "
    "flamingos from the number of pink flamingos: "
    "(36 - 6 = 30). Therefore, at noon on Sunday, there were "
    "30 more pink plastic flamingos out than white plastic "
    "flamingos. The answer is (\\boxed{30}).",
]

In [6]:
for name, system in system_prompts.items():
    scores = score_rlhflow_prm(
        model, tokenizer, problem, reasoning_steps,
        candidate_token_ids, system=system,
    )
    print(f"=== system_prompt: {name!r} ===")
    print_step_scores(reasoning_steps, scores)
    print()

=== system_prompt: 'empty' ===
Step 1: P(correct) = 0.9980
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.8174
On Saturday, they take back one third of the flamingos. Sinc...
Step 3: P(correct) = 0.9604
On Sunday, the neighbors add another 18 pink plastic flaming...
Step 4: P(correct) = 0.9829
To find the difference, subtract the number of white flaming...

=== system_prompt: 'qwen_sys_prompt' ===
Step 1: P(correct) = 0.9902
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.8501
On Saturday, they take back one third of the flamingos. Sinc...
Step 3: P(correct) = 0.9585
On Sunday, the neighbors add another 18 pink plastic flaming...
Step 4: P(correct) = 0.9854
To find the difference, subtract the number of white flaming...

=== system_prompt: 'custom_sys_prompt' ===
Step 1: P(correct) = 0.9136
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.7744
On Saturday, they take back

## Example 2 — correct vs. wrong trajectory

Equation `3x + 5 = 17`. The wrong trajectory divides by 2
instead of 3 at step 2. We check whether the system prompt
affects how sharply the PRM flags the error.

In [7]:
algebra_problem = "If 3x + 5 = 17, what is x?"

correct_steps = [
    "We need solve the equation 3x + 5 = 17.",
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 3 gives x = 4.",
    "Therefore, the answer is (\\boxed{4}).",
]

wrong_steps = [
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 2 gives x = 6.",
    "Therefore, the final answer is \\boxed{6}.",
]

In [8]:
# Show the formatted conversation for empty, qwen_sys_prompt,
# and custom_sys_prompt with the correct algebra trajectory.
for name in ("empty", "qwen_sys_prompt", "custom_sys_prompt"):
    print(f"===== sys_prompt: {name!r} =====")
    _ = score_rlhflow_prm(
        model, tokenizer,
        algebra_problem, correct_steps,
        candidate_token_ids,
        system=system_prompts[name],
        print_conversation=True,
    )
    print()

===== sys_prompt: 'empty' =====
===== Formatted PRM input =====
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

If 3x + 5 = 17, what is x? We need solve the equation 3x + 5 = 17.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

+<|eot_id|><|start_header_id|>user<|end_header_id|>

Subtracting 5 from both sides gives 3x = 12.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

+<|eot_id|><|start_header_id|>user<|end_header_id|>

Dividing both sides by 3 gives x = 4.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

+<|eot_id|><|start_header_id|>user<|end_header_id|>

Therefore, the answer is (\boxed{4}).<|eot_id|><|start_header_id|>assistant<|end_header_id|>

+<|eot_id|>

===== sys_prompt: 'qwen_sys_prompt' =====
===== Formatted PRM input =====
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Tod

In [9]:
for name, system in system_prompts.items():
    correct_scores = score_rlhflow_prm(
        model, tokenizer, algebra_problem, correct_steps,
        candidate_token_ids, system=system,
    )
    wrong_scores = score_rlhflow_prm(
        model, tokenizer, algebra_problem, wrong_steps,
        candidate_token_ids, system=system,
    )
    print(f"=== system_prompt: {name!r} ===")
    print("--- Correct trajectory ---")
    print_step_scores(correct_steps, correct_scores)
    print("--- Wrong trajectory (step 2 divides by 2) ---")
    print_step_scores(wrong_steps, wrong_scores)
    print()

=== system_prompt: 'empty' ===
--- Correct trajectory ---
Step 1: P(correct) = 0.9990
We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 1.0000
Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 1.0000
Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 0.9985
Therefore, the answer is (\boxed{4}).
--- Wrong trajectory (step 2 divides by 2) ---
Step 1: P(correct) = 0.9995
Subtracting 5 from both sides gives 3x = 12.
Step 2: P(correct) = 0.2394
Dividing both sides by 2 gives x = 6.
Step 3: P(correct) = 0.8599
Therefore, the final answer is \boxed{6}.

=== system_prompt: 'qwen_sys_prompt' ===
--- Correct trajectory ---
Step 1: P(correct) = 0.9937
We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 1.0000
Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 1.0000
Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 0.9995
Therefore, the answer is (\boxed{4}).
--- Wrong trajectory (step 2 divides by 2) ---
Step 1: P(correct) = 0.9

## Cleanup

In [10]:
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

GPU memory used: 14.35 GB
